In [11]:
import pandas as pd
import numpy as np
import sqlite3
# Conectamos con la base de datos eventos.db
connection = sqlite3.connect('eventos.db')

# Obtenemos un cursor que utilizaremos para hacer las queries
crsr = connection.cursor()
# Con esta función leemos los datos y lo pasamos a un DataFrame de Pandas
def sql_query(query):

    # Ejecuta la query
    crsr.execute(query)

    # Almacena los datos de la query 
    ans = crsr.fetchall()

    # Obtenemos los nombres de las columnas de la tabla
    names = [description[0] for description in crsr.description]

    return pd.DataFrame(ans,columns=names)


In [13]:
# Borramos tabla para pruebas
query = f'drop table if exists eventos'
crsr.execute(query);
query = f'drop table if exists families'
crsr.execute(query);
query = f'drop table if exists family_members'
crsr.execute(query);
query = f'drop table if exists user_favourite_events'
crsr.execute(query);
query = f'drop table if exists user_selected_recomendations'
crsr.execute(query);
query = f'drop table if exists users'
crsr.execute(query);

In [14]:
# Crear tabla eventos
sql_cre = f'''
CREATE TABLE eventos (
    id INTEGER PRIMARY KEY,

    business_id VARCHAR(50),
    fuente VARCHAR(255),
    external_id VARCHAR(255),

    title VARCHAR(255),
    description TEXT,

    categoria VARCHAR(100),
    tipo_plantilla VARCHAR(100),

    municipio VARCHAR(100),
    territorio VARCHAR(50),
    address VARCHAR(255),

    lat DECIMAL(10,7),
    lng DECIMAL(10,7),

    telefono VARCHAR(50),
    email VARCHAR(255),
    website VARCHAR(255),

    es_interior BOOLEAN,
    es_carrito BOOLEAN,
    es_cambiador BOOLEAN,
    es_silla_ruedas BOOLEAN,
    es_mascotas BOOLEAN,

    edad_minima INT,

    fecha_inicio DATE,
    fecha_fin DATE,

    lugar VARCHAR(255),
    price VARCHAR(50),
    imagen_url TEXT,
    tipo_evento VARCHAR(100)
);
'''

crsr.execute(sql_cre)

In [15]:
# 1. Leer CSV
df = pd.read_csv("datos/events_2026-06-02.csv", sep=";")

# 2. Reemplazar NaN por None (para que SQL los acepte como NULL)
df = df.where(pd.notnull(df), None)

# 3. Preparar columnas y placeholders
columnas = [col for col in df.columns if col != "id"]
cols_sql = ", ".join(columnas)
placeholders = ", ".join(["?"] * len(columnas))   # Si usas SQLite
# Para MySQL sería: "%s" en vez de "?"

sql_ins = f"INSERT INTO eventos ({cols_sql}) VALUES ({placeholders})"

# 4. Insertar fila a fila
for _, fila in df.iterrows():
    valores = tuple(fila[columnas])
    crsr.execute(sql_ins, valores)

# crsr.commit()
print("Inserción completada")

Inserción completada


In [16]:
# Comprobamos update
query = '''
select *
    from eventos;
'''
sql_query(query)

,id,business_id,fuente,external_id,title,description,categoria,tipo_plantilla,municipio,territorio,...,es_cambiador,es_silla_ruedas,es_mascotas,edad_minima,fecha_inicio,fecha_fin,lugar,price,imagen_url,tipo_evento
0,1,None,https://turismoa.euskadi.eus/es/centros-comerc...,Dendaraba,None,comercio,Centros comerciales,Vitoria-Gasteiz,araba,La Paz,...,1,0,0,2026-06-02,None,None,None,None,None,None
1,2,None,https://turismoa.euskadi.eus/es/centros-comerc...,El Boulevard,None,comercio,Centros comerciales,Vitoria-Gasteiz,araba,"Zaramaga, 1",...,1,0,0,2026-06-02,None,None,None,None,None,None
2,3,None,https://turismoa.euskadi.eus/es/centros-comerc...,Parque Comercial Gorbeia,None,comercio,Centros comerciales,Zigoitia,araba,"Carretera Miñano Mayor, s/n",...,1,0,0,2026-06-02,None,None,None,None,None,None
3,4,None,https://turismoa.euskadi.eus/es/centros-comerc...,Lakua Centro,None,comercio,Centros comerciales,Vitoria-Gasteiz,araba,"Duque de Wellington, s/n",...,1,0,0,2026-06-02,None,None,None,None,None,None
4,5,None,https://turismoa.euskadi.eus/es/centros-comerc...,Artea,None,comercio,Centros comerciales,Leioa,bizkaia,"Barrio Peruri, 33",...,1,0,0,2026-06-02,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6292,6293,None,https://turismoa.euskadi.eus/es/rutas/de-zalla...,"De Zalla a Balmaseda, las Encartaciones con perro",La ruta entre Zalla y Balmaseda es un bonito p...,ruta,Rutas,Zalla;Balmaseda,bizkaia,None,...,0,1,4,2026-06-02,None,None,None,None,None,None
6293,6294,None,https://turismoa.euskadi.eus/es/rutas/via-verd...,Vía Verde del Urola con perro,Esta es una ruta pensada especialmente para aq...,ruta,Rutas,Azpeitia;Legazpi;Zumarraga;Azkoitia,gipuzkoa,None,...,0,1,8,2026-06-02,None,None,None,None,None,None
6294,6295,None,https://turismoa.euskadi.eus/es/rutas/vinedos-...,Un paseo entre los viñedos de Rioja Alavesa co...,Esta es una ruta pensada especialmente para aq...,ruta,Rutas,Elciego,araba,None,...,0,1,6,2026-06-02,None,None,None,None,None,None
6295,6296,None,https://turismoa.euskadi.eus/es/rutas/lagares-...,Lagares rupestres de Labastida con perro,Esta es una ruta pensada especialmente para aq...,ruta,Rutas,Labastida/Bastida,araba,None,...,0,1,4,2026-06-02,None,None,None,None,None,None


In [17]:
connection.commit() # Commit por si nos queda algo pendiente antes de cerrar conexión
connection.close()

In [10]:
import sqlite3

try:
    conn = sqlite3.connect("eventos.db")
    print("OK: la base de datos se abre correctamente")
    conn.close()
except Exception as e:
    print("ERROR:", e)

OK: la base de datos se abre correctamente
